# 01 — Existence Matrices (Full Scale)

Loads the full plant and pollinator existence matrices and verifies
their alignment to the 3,162 common CONUS bins.

The F matrix (plant) is built in `data/full_scale_run/01_plant_flowering.ipynb`.
The P matrix (pollinator) is built in `data/full_scale_run/03_pollinator_occurrences.ipynb`.

This notebook serves as the entry point for the representation pipeline —
it loads both matrices, confirms alignment, and computes summary statistics
before the PCA step.

**Inputs:**
- `stage4_F_existence_phenofield.csv` (6,466 species × 3,162 bins)
- `stage4_P_existence_corrected.csv` (24,939 species × 3,162 bins)

**Key constraint:** All downstream N computations must use
`F_common @ P_common` where both are restricted to the same 3,162 common bins.
Never mix F and P arrays with different column sets.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE    = Path("/scratch/ariana.l")
OLD_S4  = BASE / "Stage 4 Link Prediction Model"
NEW_S4  = BASE / "New Stage 4 Link Prediction Model"

F_PATH  = OLD_S4 / "stage4_F_existence_phenofield.csv"
P_PATH  = NEW_S4 / "stage4_P_existence_corrected.csv"

print("Paths OK")

In [ ]:
# Load F and P matrices
print("Loading F matrix (plant side)...")
F_df = pd.read_csv(F_PATH, index_col=0)
print(f"  F shape    : {F_df.shape}")
print(f"  F sparsity : {1 - F_df.values.mean():.4f}")

print("\nLoading P matrix (pollinator side)...")
P_df = pd.read_csv(P_PATH, index_col=0)
print(f"  P shape    : {P_df.shape}")
print(f"  P sparsity : {1 - P_df.values.mean():.4f}")

In [ ]:
# Verify bin alignment
f_bins = set(F_df.columns)
p_bins = set(P_df.columns)
common_bins = sorted(f_bins & p_bins)

print(f"F bins:      {len(f_bins):,}")
print(f"P bins:      {len(p_bins):,}")
print(f"Common bins: {len(common_bins):,}")
# Expected: 3,162

print(f"\nBin format sample: {common_bins[:3]}")
# Expected: '24.5_-81.0' style

In [ ]:
# Restrict to common bins and build numpy arrays for fast N computation
F_common = F_df[common_bins].values   # (n_plants × 3162)
P_common = P_df[common_bins].values   # (n_pollinators × 3162)

# Index maps for fast row lookup
fc_idx = {sp: i for i, sp in enumerate(F_df.index)}
pc_idx = {sp: i for i, sp in enumerate(P_df.index)}

def compute_N(plant, pollinator):
    """Shared bin count for a (plant, pollinator) pair."""
    return float(F_common[fc_idx[plant]] @ P_common[pc_idx[pollinator]])

# Spot-check
sample_plant = F_df.index[0]
sample_poll  = P_df.index[0]
print(f"Sample N ({sample_plant} × {sample_poll}): {compute_N(sample_plant, sample_poll):.0f} shared bins")

In [ ]:
# Coverage summary
print("Coverage summary:")
print(f"  Bins with ≥1 plant species:      {(F_df[common_bins].sum(axis=0) > 0).sum():,}")
print(f"  Bins with ≥1 pollinator species: {(P_df[common_bins].sum(axis=0) > 0).sum():,}")
print(f"  Plants with ≥1 CONUS bin:        {(F_df[common_bins].sum(axis=1) > 0).sum():,}")
print(f"  Pollinators with ≥1 CONUS bin:   {(P_df[common_bins].sum(axis=1) > 0).sum():,}")